# NYC 311 Demand: Exploratory Data Analysis

Exploring 3 years (June 2023 – June 2026) of NYC 311 service request
volume across 14 high-frequency complaint categories, sourced from the
`fct_daily_demand` mart.

**Goals of this notebook:**
- Establish citywide daily volume patterns (trend, weekly/yearly seasonality)
- Identify which complaint categories show meaningfully different demand
  shapes, to justify category-level (not just aggregate) forecasting
- Surface any data issues that should inform model choice before
  modeling begins

Note: This notebook is exploratory only and no modeling happens here. Baseline
and ML models are built in `02_baseline_sarima.ipynb` onward.

## Setup

Load `fct_daily_demand` directly from the DuckDB warehouse.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

WAREHOUSE_PATH = Path.cwd().parents[1] / "pipeline" / "warehouse" / "nyc311.duckdb"
con = duckdb.connect(str(WAREHOUSE_PATH), read_only=True)

df = con.execute("SELECT * FROM fct_daily_demand ORDER BY request_date").fetchdf()
print(f"Shape: {df.shape}")
print(f"Date range: {df['request_date'].min()} to {df['request_date'].max()}")
df.head()

Shape: (77042, 13)
Date range: 2023-06-01 00:00:00 to 2026-06-18 00:00:00


,request_date,complaint_category_id,complaint_type,category_group,borough,request_count,closed_count,avg_resolution_hours,same_day_batch_closed_count,day_of_week,is_weekend,month,year
0,2023-06-01,8bc81328de7ff11a32bae31751a3c7ff,Air Quality,Environment,BRONX,1,1,99.00,0,4,False,6,2023
1,2023-06-01,8bc81328de7ff11a32bae31751a3c7ff,Air Quality,Environment,BROOKLYN,10,10,76.70,0,4,False,6,2023
2,2023-06-01,8bc81328de7ff11a32bae31751a3c7ff,Air Quality,Environment,MANHATTAN,28,28,75.25,0,4,False,6,2023
3,2023-06-01,8bc81328de7ff11a32bae31751a3c7ff,Air Quality,Environment,QUEENS,16,16,61.25,0,4,False,6,2023
4,2023-06-01,8bc81328de7ff11a32bae31751a3c7ff,Air Quality,Environment,STATEN ISLAND,1,1,4.00,0,4,False,6,2023


## Citywide daily volume

This is the aggregate series every downstream model will ultimately be compared against.
Looking specifically for: data artifacts (gaps, impossible spikes), weekly seasonality, yearly seasonality, and overall trend direction.

In [2]:
daily_total = (
    df.groupby('request_date', as_index=False)['request_count']
    .sum()
    .rename(columns={'request_count': 'total_requests'})
)

print(f"Daily total shape: {daily_total.shape}")
print(daily_total.describe())

fig = px.line(
    daily_total, x='request_date', y='total_requests',
    title='NYC 311 Total Daily Request Volume (All 14 Categories, Citywide)'
)
fig.update_layout(height=450)
fig.show()

Daily total shape: (1114, 2)
              request_date  total_requests
count                 1114     1114.000000
mean   2024-12-08 12:00:00     5593.143627
min    2023-06-01 00:00:00      306.000000
25%    2024-03-05 06:00:00     4769.250000
50%    2024-12-08 12:00:00     5337.000000
75%    2025-09-12 18:00:00     6105.750000
max    2026-06-18 00:00:00    12488.000000
std                    NaN     1213.734858


**Observations:**

- **Weekly seasonality** is strong and consistent across the full 3-year
  window — a clean weekday/weekend zigzag.

- **Upward trend**: daily volume grows from a ~4–5k baseline (2023) to
  ~6–7k (2026), consistent with year-over-year row counts already seen
  in the marts layer. Likely real demand growth, but not yet confirmed
  against category mix — flagged for follow-up.

- **Recurring spikes** (mid-2024, mid-2025, early 2026), peaking ~12.4k
  vs. a ~5.6k mean. Cause not yet identified — to be investigated at
  category level below.

- **Data artifact at the trailing edge**: final day(s) show a near-zero
  count (min = 306), almost certainly reporting lag rather than real
  demand. Will bias any model trained on it toward a false end-of-series
  collapse — trimmed in the next cell.

In [3]:
print("Last 10 days of data:")
print(daily_total.tail(10))

Last 10 days of data:
     request_date  total_requests
1104   2026-06-09            6313
1105   2026-06-10            5661
1106   2026-06-11            5915
1107   2026-06-12            6168
1108   2026-06-13            6456
1109   2026-06-14            7237
1110   2026-06-15            6067
1111   2026-06-16            5902
1112   2026-06-17            5157
1113   2026-06-18             306


## Handling the trailing partial day

Confirmed: exactly one anomalous day (`2026-06-18`, 306 requests vs.
neighboring days in the 5,000–7,200 range). Dropped dynamically using
`.max()` rather than a hardcoded date, so this fix stays correct if the
notebook is re-run later against a fresher pull.

In [4]:
cutoff_date = daily_total['request_date'].max()
print(f"Dropping trailing partial day: {cutoff_date.date()} "
      f"({daily_total.loc[daily_total['request_date'] == cutoff_date, 'total_requests'].values[0]} requests)")

daily_total = daily_total[daily_total['request_date'] < cutoff_date].copy()

print(f"New shape: {daily_total.shape}")
print(f"New date range: {daily_total['request_date'].min().date()} to {daily_total['request_date'].max().date()}")
print(daily_total.tail(5))

Dropping trailing partial day: 2026-06-18 (306 requests)
New shape: (1113, 2)
New date range: 2023-06-01 to 2026-06-17
     request_date  total_requests
1108   2026-06-13            6456
1109   2026-06-14            7237
1110   2026-06-15            6067
1111   2026-06-16            5902
1112   2026-06-17            5157


## Verifying the demand growth

Checking that the upward volume trend is real demand growth, or could it be a category-mix artifact (e.g. one category's data simply being more complete in later years than earlier years). Since 2023 and 2026 are partial years I will compare 2024 to 2025

In [5]:
yearly_by_category = (
    df.assign(year=df['request_date'].dt.year)
    .groupby(['year', 'complaint_type'], as_index=False)['request_count']
    .sum()
)

pivot = yearly_by_category.pivot(index='complaint_type', columns='year', values='request_count')
pivot['2024_to_2025_pct_change'] = ((pivot[2025] - pivot[2024]) / pivot[2024] * 100).round(1)
pivot_sorted = pivot.sort_values('2024_to_2025_pct_change', ascending=False)

print(pivot_sorted[[2024, 2025, '2024_to_2025_pct_change']])

year                        2024    2025  2024_to_2025_pct_change
complaint_type                                                   
Noise - Residential       379296  463349                     22.2
HEAT/HOT WATER            264750  315946                     19.3
Water System               65789   77518                     17.8
Illegal Parking           505728  577257                     14.1
Traffic Signal Condition   43040   47998                     11.5
Noise - Street/Sidewalk   163001  173049                      6.2
Air Quality                 9269    9800                      5.7
PLUMBING                   65904   69547                      5.5
Blocked Driveway          170192  172723                      1.5
Sewer                      26249   26141                     -0.4
Street Condition           71159   70330                     -1.2
Damaged Tree               31721   30089                     -5.1
Noise - Commercial         68347   63958                     -6.4
PAINT/PLAS

## Is the upward trend broad-based or category-specific?

Comparing the only two full, comparable years:

- **Growth is broad but uneven**, not concentrated in a single category
  (which would suggest a pipeline artifact). 9 of 14 categories grew
  year-over-year; 5 declined.
- **Largest growth**: Noise - Residential (+22.2%), HEAT/HOT WATER
  (+19.3%), Water System (+17.8%), Illegal Parking (+14.1%).
- **Largest declines**: PAINT/PLASTER (-8.0%), Noise - Commercial
  (-6.4%), Damaged Tree (-5.1%).
- This spread across both directions is reassuring: it supports treating
  the citywide upward trend as **real demand signal**, not a data
  quality issue — while also showing that a single citywide trend
  assumption would actually understate some categories and overstate
  others. This is the core justification for forecasting at the
  **category level**, not just in aggregate.

## Weekly seasonality

The zigzag pattern visible in the citywide chart is almost certainly
weekday vs. weekend cycling. Quantifying it here with a day-of-week
breakdown, since "weekly seasonality exists" needs to become a specific,
usable number for feature engineering later (e.g. LightGBM's
`day_of_week` feature, SARIMA's seasonal order).

In [6]:
daily_total['day_of_week'] = daily_total['request_date'].dt.day_name()
daily_total['dow_num'] = daily_total['request_date'].dt.dayofweek  # Monday=0

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_summary = (
    daily_total.groupby('day_of_week')['total_requests']
    .agg(['mean', 'median', 'std'])
    .reindex(dow_order)
    .round(0)
)
print(dow_summary)

fig = px.box(
    daily_total, x='day_of_week', y='total_requests',
    category_orders={'day_of_week': dow_order},
    title='Daily Request Volume Distribution by Day of Week'
)
fig.update_layout(height=450)
fig.show()

               mean  median     std
day_of_week                        
Monday       5647.0  5315.0  1298.0
Tuesday      5527.0  5165.0  1284.0
Wednesday    5321.0  5005.0  1165.0
Thursday     5333.0  5002.0  1133.0
Friday       5534.0  5299.0  1052.0
Saturday     5775.0  5565.0   991.0
Sunday       6048.0  5806.0  1319.0


**Observations:**

- **Weekends peak, not weekdays** — Sunday (mean 6,048) and Saturday
  (5,775) are highest; Wed/Thu (~5,320) lowest. Inverse of a typical
  "business hours" complaint pattern.
- Makes sense given category mix: top-volume categories (Illegal
  Parking, Noise, Blocked Driveway) are inherently weekend-skewed.
- Full distribution shifts on weekends (median, IQR, and outliers all
  higher), not just the mean.
- **Implication**: day-of-week is a real, strong feature, which is confirmed
  here, not assumed, and belongs explicitly in LightGBM's features
  and SARIMA's seasonal order.

## Monthly seasonality

Testing the hypothesis from the citywide chart: do specific months
show systematic spikes (e.g. heat complaints in winter, noise in
summer)? Aggregating across all 3 years to separate seasonal pattern
from one-off events.

In [7]:
monthly = (
    daily_total.assign(month=daily_total['request_date'].dt.month_name())
    .groupby('month')['total_requests']
    .agg(['mean', 'median'])
    .reindex(['January','February','March','April','May','June',
              'July','August','September','October','November','December'])
    .round(0)
)
print(monthly)

fig = px.box(
    daily_total.assign(month=daily_total['request_date'].dt.strftime('%m-%b')),
    x='month', y='total_requests',
    title='Daily Request Volume Distribution by Month (All Years Combined)'
)
fig.update_layout(height=450)
fig.show()

             mean  median
month                    
January    6702.0  5963.0
February   5984.0  5576.0
March      5688.0  5374.0
April      5247.0  5186.0
May        5427.0  5216.0
June       5422.0  5299.0
July       4892.0  4762.0
August     4752.0  4551.0
September  5239.0  4946.0
October    5673.0  5390.0
November   5929.0  5838.0
December   6266.0  5818.0


**Observations:**

- **Winter peaks, summer troughs** — January highest (mean 6,702),
  August lowest (4,752). A real, gradual seasonal cycle, not noise.
- Likely driven in large part by HEAT/HOT WATER — a winter-only
  emergency category, and one of the fastest-growing per the
  2024→2025 comparison.
- The earlier spikes (seen in the citywide chart) sit on top of this
  baseline as localized events, not the seasonal pattern itself —
  worth identifying their actual cause directly rather than assuming.
- **Implication**: month/season is a strong feature for both SARIMA
  and LightGBM — but category-level seasonality will likely differ
  (e.g. HEAT/HOT WATER winter-driven), supporting category-level
  forecasting over aggregate-only.

In [8]:
spike_dates = daily_total.nlargest(5, 'total_requests')[['request_date', 'total_requests']]
print("Top 5 highest-volume days:")
print(spike_dates)

for d in spike_dates['request_date']:
    day_breakdown = (
        df[df['request_date'] == d]
        .groupby('complaint_type')['request_count']
        .sum()
        .sort_values(ascending=False)
        .head(3)
    )
    print(f"\n{d.date()} top categories:")
    print(day_breakdown)

Top 5 highest-volume days:
    request_date  total_requests
976   2026-02-01           12488
983   2026-02-08           11689
584   2025-01-05           11092
472   2024-09-15           10937
588   2025-01-09           10761

2026-02-01 top categories:
complaint_type
HEAT/HOT WATER              4688
Traffic Signal Condition    2646
Illegal Parking             1688
Name: request_count, dtype: int64

2026-02-08 top categories:
complaint_type
HEAT/HOT WATER         6118
Illegal Parking        1542
Noise - Residential    1170
Name: request_count, dtype: int64

2025-01-05 top categories:
complaint_type
Noise - Residential    6053
HEAT/HOT WATER         2205
Illegal Parking        1392
Name: request_count, dtype: int64

2024-09-15 top categories:
complaint_type
Noise - Residential        6559
Illegal Parking            1564
Noise - Street/Sidewalk    1331
Name: request_count, dtype: int64

2025-01-09 top categories:
complaint_type
Noise - Residential    3620
HEAT/HOT WATER         3599
Illeg

In [9]:
sept_15 = df[df['request_date'] == '2024-09-15'].groupby('complaint_type')['request_count'].sum().sort_values(ascending=False).head(5)
print("2024-09-15 breakdown:")
print(sept_15)

2024-09-15 breakdown:
complaint_type
Noise - Residential        6559
Illegal Parking            1564
Noise - Street/Sidewalk    1331
Blocked Driveway            527
Noise - Commercial          367
Name: request_count, dtype: int64


**Spike investigation:**

- **4 of the top 5 highest-volume days are HEAT/HOT WATER-driven**,
  all in Jan/Feb (e.g. 2026-02-01: 12,488 total, 4,688 HEAT/HOT
  WATER alone). Confirms winter seasonality is partly **spiky and
  threshold-driven** (cold snaps), not just a smooth curve.
- **The 5th (2024-09-15) is Noise-driven** (6,559 Noise-Residential
  alone) — a warm-weather Sunday, consistent with the weekend/summer
  Noise pattern already established above. Not an artifact.
- **Implication**: HEAT/HOT WATER and Noise spike for different
  reasons on different schedules — supports modeling them separately
  with category-specific features (e.g. cold-snap sensitivity for
  HEAT/HOT WATER) rather than one shared seasonal model.

## Category-level seasonality comparison

Selected 4 categories with genuinely different demand shapes, based on
findings above:
- **HEAT/HOT WATER** — winter-spiky, threshold-driven
- **Noise - Residential** — weekend/summer-skewed
- **Illegal Parking** — highest volume, steady
- **Air Quality** — lowest volume, sparse

Comparing these directly justifies category-level forecasting over a
single aggregate model — if their shapes were similar, aggregate
forecasting would suffice.

In [10]:
selected_categories = ['HEAT/HOT WATER', 'Noise - Residential', 'Illegal Parking', 'Air Quality']

category_daily = (
    df[df['complaint_type'].isin(selected_categories)]
    .groupby(['request_date', 'complaint_type'], as_index=False)['request_count']
    .sum()
)

# Apply the same trailing-partial-day trim established earlier
category_daily = category_daily[category_daily['request_date'] < cutoff_date].copy()

fig = px.line(
    category_daily, x='request_date', y='request_count', color='complaint_type',
    title='Daily Volume by Category: HEAT/HOT WATER vs. Noise vs. Illegal Parking vs. Air Quality',
    facet_row='complaint_type'
)
fig.update_yaxes(matches=None)  # let each category have its own y-scale -- volumes differ by 100x
fig.update_layout(height=800)
fig.show()

**Observations:**

- **Four genuinely distinct shapes confirmed:**
  - *HEAT/HOT WATER*: near-zero ~6 months/year, then sharp repeating
    winter peaks — clean seasonal signal, growing year-over-year
    (~5k peak in 2023-24 → ~6k+ in 2025-26).
  - *Air Quality*: noisy, low-volume, no clear seasonal cycle —
    closer to event-driven than seasonal.
  - *Illegal Parking*: steady high baseline, gradual upward trend,
    weekly noise only — no strong seasonal swing.
  - *Noise - Residential*: tight weekly baseline + irregular sharp
    summer spikes (likely heat-wave events), distinct from HEAT/HOT
    WATER's smooth seasonal curve.

- **This directly validates category-level forecasting**: a single
  aggregate model couldn't capture both HEAT/HOT WATER's extreme
  on/off seasonality and Air Quality's near-random pattern with the
  same parameters.
- **Implication for model selection**: HEAT/HOT WATER is a strong
  Prophet/SARIMA candidate (clean seasonality). Air Quality may
  need wider uncertainty bounds or a simpler baseline — high-rigor
  modeling has limited signal to work with here.

## Summary

**Key findings:**
1. Citywide volume trends upward 2023→2026, with broad (not
   artifact-driven) category growth, confirmed at the 2024 vs. 2025
   year level.
2. Weekend volume exceeds weekday — inverse of a typical "business
   hours" pattern, driven by category mix (Noise, Illegal Parking).
3. Strong winter seasonality citywide, traced specifically to
   HEAT/HOT WATER — a near-binary seasonal category (on in winter,
   off in summer).
4. Four sampled categories show materially different demand shapes
   (clean seasonal, noisy/event-driven, steady-trending, baseline +
   spikes) — direct evidence that category-level forecasting is
   necessary.
5. One real data artifact found and fixed (trailing partial-day
   reporting lag).

**Carrying forward into modeling:**
- HEAT/HOT WATER and Noise - Residential are strong candidates for
  category-level Prophet models given their clear, distinct seasonal
  structure.
- Air Quality may need a simpler baseline or wider uncertainty
  bounds — limited seasonal signal to exploit.
- day_of_week and month are confirmed, validated features for
  LightGBM — not assumed, derived from this analysis.